<a href="https://colab.research.google.com/github/joexner/roxene/blob/master/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd
# Check if roxene directory exists, if not, clone it.
!if [ ! -d roxene ]; then git clone https://github.com/joexner/roxene.git; fi

In [ ]:
# Change directory to roxene and pull latest changes.
%cd ~/roxene
!git pull
!git log -1 --pretty="%ci: %s"

In [ ]:
!pip install -e .

In [ ]:
from google.colab import auth


# Authenticate to GCP
auth.authenticate_user()


# Set your GCP Project ID and Instance details
PROJECT_ID = 'roxene-0'
!gcloud config set project {PROJECT_ID}


# Use a static instance name for the long-running instance
INSTANCE_NAME = 'roxene-test'
DB_PASSWORD = 'enexor'


In [ ]:

import uuid
import subprocess

print(f"Checking for Cloud SQL instance: {INSTANCE_NAME}")

# Check if instance already exists
instance_exists = subprocess.run(
    f"gcloud sql instances describe {INSTANCE_NAME}",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
).returncode == 0

if not instance_exists:
    print(f"Creating Cloud SQL instance: {INSTANCE_NAME} (this may take a few minutes)...")
    !gcloud sql instances create {INSTANCE_NAME} \
        --database-version=POSTGRES_14 \
        --cpu=2 --memory=8GB \
        --region=us-central1 \
        --root-password={DB_PASSWORD} \
        --assign-ip

else:
    print(f"Instance {INSTANCE_NAME} already exists. skipping creation.")

# Always get the public IP address of the instance (needed for connection URL later)
output = subprocess.check_output(
    f"gcloud sql instances describe {INSTANCE_NAME} --format='value(ipAddresses[0].ipAddress)'",
    shell=True, text=True
).strip()
PUBLIC_IP = output
print(f"Database Public IP: {PUBLIC_IP}")

In [ ]:
# Get Colab's public IP
colab_ip = subprocess.check_output("curl -s ifconfig.me", shell=True, text=True).strip()
print(f"Colab Public IP: {colab_ip}")

# Authorize Colab's IP to allow connections
print("Adding Colab IP to authorized networks...")
!gcloud sql instances patch {INSTANCE_NAME} --authorized-networks={colab_ip} --quiet

In [ ]:
import subprocess
import uuid

# Generate a random suffix to ensure database name uniqueness per run
random_suffix = uuid.uuid4().hex[:6]
DB_NAME = f'roxene_{random_suffix}'

print(f"Creating new database: {DB_NAME}")
# Create the database inside the Cloud SQL instance
!gcloud sql databases create {DB_NAME} --instance={INSTANCE_NAME}

In [ ]:
# Construct the DB URL and run the script
db_url = f"postgresql+psycopg2://postgres:{DB_PASSWORD}@{PUBLIC_IP}:5432/{DB_NAME}"
print(f"Connecting to: {db_url}")

!python -m roxene.tic_tac_toe 1000 10000 --num_threads=20 --db_url={db_url}